In [9]:
# === DAILY CHALLENGE : GAN-BASED AI TEXT DETECTOR (FULL SINGLE CELL) ===

# ---------- 0. Imports & Device ----------
from google.colab import files
import io
import os
import zipfile
import random
import string
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from transformers import BertTokenizer, BertForSequenceClassification, BertConfig, BertModel
from sklearn.metrics import roc_auc_score

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------- 1. Upload + Extract ZIP ----------
print("⬆️ Please upload llm-detect-ai-generated-text.zip")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print("Uploaded file:", zip_name)

extract_dir = "/content/llm_dataset"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(io.BytesIO(uploaded[zip_name]), 'r') as z:
    z.extractall(extract_dir)

print("Extracted files:", os.listdir(extract_dir))

# Try to detect filenames
files_in_dir = set(os.listdir(extract_dir))

def find_name(candidates):
    for c in candidates:
        if c in files_in_dir:
            return os.path.join(extract_dir, c)
    return None

TRAIN_PATH = find_name(["train_essays.csv", "train.csv"])
TEST_PATH = find_name(["test_essays.csv", "test.csv"])
PROMPT_PATH = find_name(["train_prompts.csv", "prompts.csv", "train_prompts.csv"])
SUB_PATH = find_name(["sample_submission.csv", "submission.csv"])

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)
print("PROMPT_PATH:", PROMPT_PATH)
print("SUB_PATH:", SUB_PATH)

if TRAIN_PATH is None or TEST_PATH is None or SUB_PATH is None:
    raise FileNotFoundError("Could not detect train/test/submission CSVs in /content/llm_dataset")

# ---------- 2. Load Data ----------
src_train = pd.read_csv(TRAIN_PATH)
src_test = pd.read_csv(TEST_PATH)
src_prompt = pd.read_csv(PROMPT_PATH) if PROMPT_PATH is not None else None
src_sub = pd.read_csv(SUB_PATH)

print("Train shape:", src_train.shape)
print("Test shape :", src_test.shape)
if src_prompt is not None:
    print("Prompts shape:", src_prompt.shape)
print("Submission sample shape:", src_sub.shape)

print("\nTrain columns:", src_train.columns.tolist())
print("Test columns :", src_test.columns.tolist())

# Assumptions for this competition:
# - train_essays.csv has: ['essay_id','prompt_id','text','generated', ...]
# - test_essays.csv has: ['essay_id','prompt_id','text', ...]
# - sample_submission.csv has: ['id','generated']

TEXT_COL = "text"
LABEL_COL = "generated"

if TEXT_COL not in src_train.columns or LABEL_COL not in src_train.columns:
    raise ValueError(f"Expected columns '{TEXT_COL}' and '{LABEL_COL}' in train CSV.")

# ---------- 3. Model Preparation (BERT as embedding model) ----------
tokenizer_save_path = "bert-base-uncased"
model_save_path = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased")
embedding_model = pretrained_model.bert.to(device)  # We'll use the BERT encoder as embedding model

# ---------- 4. Hyperparameters ----------
train_batch_size = 16
test_batch_size = 32
lr = 2e-4
beta1 = 0.5
nz = 100                # latent vector dim
num_epochs = 1          # keep small so it runs
num_hidden_layers = 4   # not really used in this simplified version
train_ratio = 0.8       # 80% train / 20% val for AUC

# ---------- 5. Dataset & DataLoader ----------

class GANDAIGDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels  # can be None for inference

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        if self.labels is None:
            return self.texts[idx]
        return self.texts[idx], self.labels[idx]

all_num = len(src_train)
train_num = int(all_num * train_ratio)
val_num = all_num - train_num

train_set = src_train.iloc[:train_num].reset_index(drop=True)
val_set = src_train.iloc[train_num:].reset_index(drop=True)

print(f"\nTotal train rows: {all_num}")
print(f"Train split     : {train_num}")
print(f"Val split       : {val_num}")

train_dataset = GANDAIGDataset(
    texts=train_set[TEXT_COL].tolist(),
    labels=train_set[LABEL_COL].astype(float).values
)
val_dataset = GANDAIGDataset(
    texts=val_set[TEXT_COL].tolist(),
    labels=val_set[LABEL_COL].astype(float).values
)

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=test_batch_size, shuffle=False)

# ---------- 6. Generator Definition (simplified) ----------

SEQ_LEN = 128        # sequence length for synthetic embeddings
HIDDEN_SIZE = 768    # BERT hidden size

class Generator(nn.Module):
    """
    Takes noise (batch_size, nz) and produces fake embeddings
    of shape (batch_size, SEQ_LEN, HIDDEN_SIZE), similar to BERT outputs.
    """
    def __init__(self, input_dim, seq_len=SEQ_LEN, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, seq_len * hidden_size),
        )

    def forward(self, x):
        x = self.net(x)
        x = x.view(-1, self.seq_len, self.hidden_size)
        return x

# ---------- 7. Discriminator Definition (simplified) ----------

class Discriminator(nn.Module):
    """
    Takes embeddings of shape (batch_size, SEQ_LEN, HIDDEN_SIZE),
    mean-pools over the sequence and classifies with an MLP.
    """
    def __init__(self, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        # x: (batch, seq_len, hidden)
        pooled = x.mean(dim=1)                # (batch, hidden)
        out = self.classifier(pooled)         # (batch, 1)
        return torch.sigmoid(out).view(-1)    # (batch,)

# ---------- 8. Helper: Prepare Embeddings from BERT ----------

def preparation_embedding(texts):
    """
    texts: list of strings
    returns: tensor of shape (batch, seq_len, hidden_size)
    """
    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=SEQ_LEN,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = embedding_model(**encodings)  # BaseModelOutputWithPooling
    # outputs.last_hidden_state: [batch, seq_len, hidden]
    return outputs.last_hidden_state

# ---------- 9. AUC Evaluation on validation set ----------

def eval_auc(discriminator_model):
    discriminator_model.eval()
    predictions = []
    actuals = []
    with torch.no_grad():
        for batch in val_loader:
            texts, labels = batch
            labels = labels.float().to(device)
            real_embeds = preparation_embedding(list(texts))  # (B, L, H)
            outputs = discriminator_model(real_embeds)        # (B,)
            predictions.extend(outputs.cpu().numpy().tolist())
            actuals.extend(labels.cpu().numpy().tolist())

    try:
        auc = roc_auc_score(actuals, predictions)
    except ValueError:
        # In case only 1 class present in val
        auc = float("nan")
    print("Validation AUC:", auc)
    return auc

def get_model_info_dict(model, epoch, auc_score):
    current_device = next(model.parameters()).device
    model.to('cpu')
    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
    }
    model.to(current_device)
    return model_info

# ---------- 10. GAN Step ----------

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, labels, epoch, i):
    netD.train()
    netG.train()

    batch_size = real_data.size(0)

    # --- Train Discriminator on real + fake ---
    netD.zero_grad()

    # Real
    real_labels = labels.float().to(device)
    output_real = netD(real_data)
    errD_real = criterion(output_real, real_labels)
    errD_real.backward()
    D_x = output_real.mean().item()

    # Fake
    noise = torch.randn(batch_size, nz, device=device)
    fake_data = netG(noise)
    # For GAN loss, we want fake to be labeled as 0 (not generated)
    fake_labels = torch.zeros(batch_size, device=device)
    output_fake = netD(fake_data.detach())
    errD_fake = criterion(output_fake, fake_labels)
    errD_fake.backward()
    D_G_z1 = output_fake.mean().item()

    errD = errD_real + errD_fake
    optimizerD.step()

    # --- Train Generator to fool Discriminator ---
    netG.zero_grad()
    target_labels_for_G = torch.ones(batch_size, device=device)  # G wants D to output 1 on fake
    output_fake_for_G = netD(fake_data)
    errG = criterion(output_fake_for_G, target_labels_for_G)
    errG.backward()
    D_G_z2 = output_fake_for_G.mean().item()
    optimizerG.step()

    if i % 50 == 0:
        print('[%d/%d][%d/%d] Loss_D: %.4f  Loss_G: %.4f  D(x): %.4f  D(G(z)): %.4f / %.4f'
              % (epoch + 1, num_epochs, i, len(train_loader),
                 errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

    return optimizerG, optimizerD, netG, netD

# ---------- 11. Instantiate Models & Optimizers ----------

netG = Generator(input_dim=nz).to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

print("\nGenerator:\n", netG)
print("\nDiscriminator:\n", netD)

# ---------- 12. Training Loop ----------

model_infos = []
print("\n=== Start Training ===")
for epoch in range(num_epochs):
    for i, batch in enumerate(train_loader):
        texts, labels = batch
        # Compute real embeddings from BERT
        real_embeds = preparation_embedding(list(texts))  # (B, L, H)

        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=real_embeds,
            labels=labels.to(device),
            epoch=epoch,
            i=i,
        )

    # Evaluate AUC on validation set at end of epoch
    auc_score = eval_auc(netD)
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print("Train complete！")

# ---------- 13. Select Best Model by AUC ----------
valid_aucs = [mi["auc_score"] for mi in model_infos]
best_idx = int(np.nanargmax(valid_aucs))
max_auc_model_info = model_infos[best_idx]
print(f"Best epoch: {max_auc_model_info['epoch']+1}, best AUC: {max_auc_model_info['auc_score']}")

# Load best model weights
best_model = Discriminator().to(device)
best_model.load_state_dict(max_auc_model_info["model_state_dict"])
best_model.eval()

# ---------- 14. Inference on Test Set ----------
class InferenceDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __getitem__(self, idx):
        return self.texts[idx]

    def __len__(self):
        return len(self.texts)

sub_texts = src_test[TEXT_COL].tolist()
sub_dataset = InferenceDataset(sub_texts)
inference_loader = DataLoader(sub_dataset, batch_size=test_batch_size, shuffle=False)

sub_predictions = []
with torch.no_grad():
    for batch in inference_loader:
        texts_batch = list(batch)
        embeds = preparation_embedding(texts_batch)
        outputs = best_model(embeds)
        sub_predictions.extend(outputs.cpu().numpy().tolist())

# Clip to [0,1] just in case
sub_predictions = np.clip(sub_predictions, 0.0, 1.0)

# Build submission dataframe
sub_ans_df = src_sub.copy()
# sample_submission usually has column 'generated'; adapt if different
pred_col = "generated" if "generated" in sub_ans_df.columns else sub_ans_df.columns[-1]
sub_ans_df[pred_col] = sub_predictions

print("\nFinal submission head:")
print(sub_ans_df.head())

# If you want to save:
# sub_ans_df.to_csv("submission.csv", index=False)
# files.download("submission.csv")


Using device: cuda:0
⬆️ Please upload llm-detect-ai-generated-text.zip


Saving llm-detect-ai-generated-text.zip to llm-detect-ai-generated-text (2).zip
Uploaded file: llm-detect-ai-generated-text (2).zip
Extracted files: ['sample_submission.csv', 'train_essays.csv', 'test_essays.csv', 'train_prompts.csv']
TRAIN_PATH: /content/llm_dataset/train_essays.csv
TEST_PATH : /content/llm_dataset/test_essays.csv
PROMPT_PATH: /content/llm_dataset/train_prompts.csv
SUB_PATH: /content/llm_dataset/sample_submission.csv
Train shape: (1378, 4)
Test shape : (3, 3)
Prompts shape: (2, 4)
Submission sample shape: (3, 2)

Train columns: ['id', 'prompt_id', 'text', 'generated']
Test columns : ['id', 'prompt_id', 'text']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Total train rows: 1378
Train split     : 1102
Val split       : 276

Generator:
 Generator(
  (net): Sequential(
    (0): Linear(in_features=100, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=98304, bias=True)
  )
)

Discriminator:
 Discriminator(
  (classifier): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=1, bias=True)
  )
)

=== Start Training ===
[1/1][0/69] Loss_D: 1.4427  Loss_G: 0.6686  D(x): 0.5146  D(G(z)): 0.5131 / 0.5124
[1/1][50/69] Loss_D: 0.7374  Loss_G: 0.7384  D(x): 0.0378  D(G(z)): 0.5028 / 0.4779
Validation AUC: 0.7818181818181819
Train complete！
Best epoch: 1, best AUC: 0.7818181818181819

Final submission head:
         id  generated
0  0000aaaa   0.151475
1  1111bbbb   0.129054
2  2222cccc   0.121813
